In [ ]:
%matplotlib ipympl

import h5py
import matplotlib.pyplot as plt
import numba
import numpy as np
import obspy
import os
import scipy.signal
import scipy.spatial
import tqdm

DTYPE_INT  = np.int32
DTYPE_REAL = np.float64

In [ ]:
# Data are assumed row vectors!
PROGRESS_BAR_MESSAGE_LENGTH = 20
PROGRESS_BAR_TOTAL_LENGTH = 80

class FastMap(object):


    def __init__(self, path, kdim, distance, sampling_rate=100, mode="w", overwrite=False):
        self._kdim = kdim
        self._sampling_rate = sampling_rate
        self._init_hdf5(path, mode, overwrite=overwrite)
        self._library_size = 0
        self._kdtree = None
        group = self.hdf5.require_group("waveforms")
        self._waveforms = WaveformGroup(group.id)
        self._distance = distance

    
    @property
    def image(self):
        return (
            self.hdf5.require_dataset(
                "/image",
                shape=(self.library_size, self.kdim),
                maxshape=(None, self.kdim),
                dtype=DTYPE_REAL,
                fillvalue=np.nan
            )
        )

    @property
    def kdim(self):
        return (self._kdim)
    
    @property
    def kdtree(self):
        if self._kdtree == None:
            self._kdtree = scipy.spatial.cKDTree(self.image)
        return (self._kdtree)
    
    @property
    def hdf5(self):
        return (self._hdf5)
    
    @property
    def library_size(self):
        """
        [Read only] The number of time series in the library.
        """
        return (self._library_size)
    
    @property
    def pivot(self):
        return (
            self.hdf5.require_dataset(
                "/pivot", 
                shape=(2, self.kdim),
                dtype=DTYPE_INT
            )
        )
    
    @property
    def sampling_rate(self):
        return (self._sampling_rate)
    
    @property
    def waveforms(self):
        return (self._waveforms)


    def __del__(self):
        self.hdf5.close()


    def __enter__(self):
        return (self)


    def __exit__(self, exc_type, exc_value, exc_traceback):
        pass
    
    def _append(self, data, labels, sampling_rate):
        """
        Append data to the library inventory. This does not perform the
        actual embedding into k-dimensional Euclidean space.
        
        Arguments
        data - A single np.ndarray.
        """
        
        id = str(self.library_size)
        self._library_size += 1
        dataset = self.waveforms.create_dataset(id, data=data)
        dataset.attrs["labels"] = labels
        
        return (True)
    
    
    def _embed(self, ik):
        """
        Recursive function to embed the library data into k-dimensional
        Euclidean space.
        """

        if ik >= self.kdim:
            return (True)
        
        print(f"Embedding dimension {ik + 1}.")
        
        # Choose the pivot objects.
        keys = list(self.waveforms.keys())
        b_name = np.random.choice(keys)
        b = self.waveforms[b_name]
    
        a_name_last = None
        b_name_last = b_name
        for i in range(8):
            a_name = self.furthest(b, k=ik)
            a = self.waveforms[a_name]
            b_name = self.furthest(a, k=ik)
            b = self.waveforms[b_name]
            if a_name == a_name_last and b_name == b_name_last:
                break
            a_name_last = a_name
            b_name_last = b_name
            
        print(f"Pivots are {a_name} and {b_name}: {distance(a, b), self.distance(a, b, k=ik)}")
        
        # Record the names of the pivot objects.
        self.pivot[0, ik] = int(a_name)
        self.pivot[1, ik] = int(b_name)
        
        # Project all objects onto line between objects a and b.
        d_ab = self.distance(a, b, k=ik)
        
        if d_ab == 0:
            self.image[:, icol] = 0
            return (True)
        
        def update_image(name, i, ik=ik):
            irow = int(name)
            d_ai = self.distance(a, i, k=ik)
            d_bi = self.distance(b, i, k=ik)
            xi = (d_ai**2 + d_ab**2 - d_bi**2) / (2 * d_ab)
            self.image[irow, ik] = xi
        
        self.waveforms.visititems(update_image)
        
        return (self._embed(ik+1))
        
    
    def _init_hdf5(self, path, mode, overwrite=False):
        """
        Initialize the HDF5 backend.
        """
        
        if os.path.exists(path) and mode == "w" and not overwrite:
            raise (IOError(f"{path} already exists."))
        self._hdf5 = h5py.File(path, mode=mode)
        

    def append(self, data, labels, sampling_rate):
        """
        Append data to the library inventory. This does not perform the
        actual embedding into k-dimensional Euclidean space.
        
        Arguments
        data - A single np.ndarray or a list of np.ndarrays.
        """
        
        # Extend this to allow for multiple pieces of data to be added
        # with one call.
        self._append(data, labels, sampling_rate)
        
        return (True)
    
    
    def distance(self, a, b, *args, k=0, **kwargs):
        
        dist = self._distance(a, b, *args, **kwargs)
        
        i = int(a.name.split("/")[-1])
        j = int(b.name.split("/")[-1])
            
        for ik in range(k):
            xa = self.image[i, ik]
            xb = self.image[j, ik]
            dist = dist**2 - (xa - xb)**2
            
            # This is a hack for distance functions that don't satisfy
            # the triangle inequality.
            dist = max(dist, 0)
            
            dist = np.sqrt(dist)
            
        return (dist)


    def embed(self):
        """
        Embed the library data into k-dimensional Euclidean space.
        """

        return_value = self._embed(0)
        return (return_value)

    
    def furthest(self, b, k):
        """
        Return the name of the object furthest from b.
        """
        
        self._furthest_name, self._furthest_dist = None, 0
        
        def _furthest(name, a, b=b, k=k):
            d = self.distance(a, b, k=k)
            if d > self._furthest_dist:
                self._furthest_name = name
                self._furthest_dist = d

        self.waveforms.visititems(_furthest)
        
        furthest_name = self._furthest_name
        del (self._furthest_name, self._furthest_dist)
        return (furthest_name)
    
   
    def query(self, q, *args, **kwargs):
        
        image = np.zeros(self.kdim,)
        
        for ik in range(self.kdim):

            # Retrieve the names of the pivot objects.
            a_name = self.pivot[0, ik] 
            b_name = self.pivot[1, ik]
            
            # Retrieve the pivot objects
            a = self.waveforms[a_name]
            b = self.waveforms[b_name]

            d_ab = self.distance(a, b, k=ik)
            
            if d_ab == 0:
                image[ik] = 0
                continue

            # Project query object onto line between objects a and b.
            d_aq = self.distance(a, q, k=ik)
            d_bq = self.distance(b, q, k=ik)
            xq = (d_aq**2 + d_ab**2 - d_bq**2) / (2 * d_ab)
            image[ik] = xq

        return (self.kdtree.query(image, *args, **kwargs))
    


class WaveformGroup(h5py.Group):
    """
    Accessor class to abstract waveform access pattern.

    This provides functionality to access waveforms using integer
    indices.
    """
    def __getitem__(self, key):
        return(super().__getitem__(str(key)))
        

def correlate(a, b):
    """
    Return the normalized cross-correlation of a and b.
    
    a and b must be either 1D arrays or 2D arrays with shapes 
    (NDIM, NPTS_A) and (NDIM, NPTS_B).
    """

    a = np.atleast_2d(a)
    b = np.atleast_2d(b)
    
    a_mean = np.mean(a, axis=1)
    b_mean = np.mean(b, axis=1)
    a_mean = np.atleast_2d(a_mean).T
    b_mean = np.atleast_2d(b_mean).T
    
    a_std = np.std(a, axis=1)
    b_std = np.std(b, axis=1)
    a_std = np.atleast_2d(a_std).T
    b_std = np.atleast_2d(b_std).T
    
    a = (a - a_mean) / (a_std * a.shape[1])
    b = (b - b_mean) / (b_std)
    
    corr = np.empty((a.shape[0], a.shape[1]+b.shape[1]-1))
    for i in range(a.shape[0]):
        corr[i] = np.correlate(a[i], b[i], "full")
            
    return (corr)

def distance(a, b):
    """
    Return the distance between a and b.
    """
    
    corr = correlate(a, b)
    cmax = np.max(corr, axis=1)
    ndim = corr.shape[0]
    dist = (ndim - np.sum(cmax)) / ndim
    
    return (dist)

In [ ]:
dataframe = pd.read_csv("/home/malcolmw/google_drive/malcolm.white@usc.edu/data/wfs/chunk2.csv")
dataframe = dataframe.set_index(["network_code", "receiver_code"])
dataframe = dataframe.sort_index()
dataframe = dataframe.loc[("TA", "109C")]

try:
    fastmap.hdf5.close()
except:
    pass

fastmap = FastMap("fastmap_test.h5", 4, distance, overwrite=True)

with h5py.File("/home/malcolmw/google_drive/malcolm.white@usc.edu/data/wfs/chunk2.hdf5", mode="r") as f5:
    for _, event in dataframe.iloc[:8].iterrows():
        trace_name = event["trace_name"]
        handle = f"data/{trace_name}"
        dataset = f5[handle]
        attrs = dataset.attrs
        for phase in ("P", "S"):
            idx = attrs[f"{phase.lower()}_arrival_sample"]
            idx_start = int(idx - 50)
            idx_end = int(idx + 100)
            trace = dataset[idx_start: idx_end].T
            fastmap._append(trace, phase, 100)
    
%time fastmap.embed()

In [ ]:
for idx in range(len(fastmap.waveforms)):
    print(fastmap.query(fastmap.waveforms[idx]))

In [ ]:
true, false = 0, 0
for idx in range(len(fastmap.waveforms)):
    q = fastmap.waveforms[idx]
    d, i = fastmap.query(q)
#     l = [fastmap.waveforms[idx].attrs["labels"] for idx in i]
    label = fastmap.waveforms[i].attrs["labels"]
    if q.attrs["labels"] == label:
        true += 1
    else:
        false += 1
true / (true+false), false/(true+false)

In [ ]:
idx, i = 1, 14
idx = np.random.choice(np.arange(len(fastmap.waveforms)))
q = fastmap.waveforms[idx]
%time d, i = fastmap.query(q)
m = fastmap.waveforms[i]


plt.close("all")
fig = plt.figure()
ax = fig.add_subplot(2, 1, 1)
ax.plot(q[:].T)
ax = fig.add_subplot(2, 1, 2)
ax.plot(m[:].T)

# ax = fig.add_subplot(3, 1, 3)
# ax.plot(q[:].T - m[:].T, "--")

# print(idx, i, q.attrs["label"], m.attrs["label"])

In [ ]:
import mpl_toolkits.mplot3d.axes3d

plt.close("all")
fig = plt.figure()
ax = fig.add_subplot(1, 1, 1, projection="3d")
for label in ("P", "S"):
    idxs = [int(key) for key in fastmap.waveforms.keys() if fastmap.waveforms[key].attrs["labels"] == label]
    idxs = sorted(idxs)
    ax.scatter(
        fastmap.image[idxs, 0],
        fastmap.image[idxs, 1],
        fastmap.image[idxs, 2],
        s=4
        
)

In [ ]:
plt.close("all")
fig = plt.figure()
ax = fig.add_subplot(1, 1, 1)
for label in ("P", "S"):
    idxs = [int(key) for key in fastmap.waveforms.keys() if fastmap.waveforms[key].attrs["labels"] == label]
    idxs = sorted(idxs)
    ax.scatter(
        fastmap.image[idxs, 0],
        fastmap.image[idxs, 1],
#         np.zeros_like(fastmap.image[idxs, 0]),
        s=16
        
)